In [1]:
import yfinance as yf
import numpy as np
tickers = ["MMM", "ABT", "ADBE", "AFL", "APD", "AEP", "AXP", "BA", "BMY", "CAT"]

data = yf.download(tickers , period='20y')['Close']

/tmp/ipykernel_188/4104306401.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers , period='20y')['Close']
[*********************100%***********************]  10 of 10 completed


In [2]:
n = len(data)
train_data = np.log(data[: int(0.7* n)]).diff().dropna().values
val_data =  np.log(data[int(0.7* n) : int(0.9*n)]).diff().dropna().values
test_data =  np.log(data[int(0.9*n) : ]).diff().dropna().values


In [3]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class FinancialDataset(Dataset):
  def __init__(self , numpy_dataset , window_size = 10) -> None:
    self.data_set = numpy_dataset.astype(np.float32)
    self.window_size = window_size

  def __len__(self):
     return len(self.data_set) - self.window_size

  def __getitem__(self, index):
     x = self.data_set[index : index + self.window_size]
     y = self.data_set[index + self.window_size]
     return x, y

train_dataset = FinancialDataset(train_data , window_size=20)
train_data_loader = DataLoader(train_dataset , batch_size= 64 , shuffle=True)


val_dataset = FinancialDataset(val_data , window_size=20)
val_data_loader = DataLoader(val_dataset , batch_size= 64 , shuffle=False)

test_dataset = FinancialDataset(test_data , window_size=20)
test_data_loader = DataLoader(test_dataset , batch_size= 64 , shuffle=False)

In [5]:
import torch.nn as nn

class FinancialLSTM(nn.Module):
  def __init__(self, indim , out_dim, hidden_dim = 20 ) -> None:
    super().__init__()
    # to do
    self.lstm = nn.LSTM(input_size=indim , hidden_size= hidden_dim , num_layers= 2 , batch_first=True)
    self.head = nn.Linear(hidden_dim , out_dim)

  def forward(self, x):
    x , _  = self.lstm(x)
    x = self.head(x[: , -1 , : ])
    return x

In [9]:
def train(
    model,
    train_data_loader,
    val_data_loader=None,
    epochs=20,
    lr=1e-3,
    weight_decay=1e-4,
    grad_clip=1.0,
    device=None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {
        "train_rmse": [],
        "val_rmse": [],
    }

    best_state = None
    best_val = float("inf")

    for epoch in range(1, epochs + 1):
        # ---------- train ----------
        model.train()
        train_loss_sum = 0.0
        train_total = 0

        for x, y in train_data_loader:
            x = x.to(device).float()
            y = y.to(device).float()

            optimizer.zero_grad()
            y_pred = model(x)
            loss = criterion(y_pred, y)
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            train_loss_sum += loss.item() * x.size(0)
            train_total += x.size(0)

        train_mse = train_loss_sum / max(train_total, 1)
        train_rmse = float(np.sqrt(train_mse))
        history["train_rmse"].append(train_rmse)

        # ---------- validation ----------
        if val_data_loader is not None:
            model.eval()
            val_loss_sum = 0.0
            val_total = 0

            with torch.no_grad():
                for x, y in val_data_loader:
                    x = x.to(device).float()
                    y = y.to(device).float()

                    y_pred = model(x)
                    loss = criterion(y_pred, y)

                    val_loss_sum += loss.item() * x.size(0)
                    val_total += x.size(0)

            val_mse = val_loss_sum / max(val_total, 1)
            val_rmse = float(np.sqrt(val_mse))
            history["val_rmse"].append(val_rmse)

            if val_rmse < best_val:
                best_val = val_rmse
                best_state = copy.deepcopy(model.state_dict())

            print(
                f"Epoch {epoch:03d}/{epochs} | "
                f"train RMSE {train_rmse:.6f} | "
                f"val RMSE {val_rmse:.6f}"
            )
        else:
            print(
                f"Epoch {epoch:03d}/{epochs} | "
                f"train RMSE {train_rmse:.6f}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

In [ ]:
model = FinancialLSTM(10 , 10, 20)

train(model, train_data_loader , val_data_loader=val_data_loader)

Epoch 001/20 | train RMSE 0.081877 | val RMSE 0.019991
Epoch 002/20 | train RMSE 0.019730 | val RMSE 0.018604
Epoch 003/20 | train RMSE 0.019367 | val RMSE 0.018608
Epoch 004/20 | train RMSE 0.019332 | val RMSE 0.018586
Epoch 005/20 | train RMSE 0.019271 | val RMSE 0.018641
Epoch 006/20 | train RMSE 0.019279 | val RMSE 0.018590
Epoch 007/20 | train RMSE 0.019207 | val RMSE 0.018606
Epoch 008/20 | train RMSE 0.019182 | val RMSE 0.018603
Epoch 009/20 | train RMSE 0.019183 | val RMSE 0.018627
Epoch 010/20 | train RMSE 0.019165 | val RMSE 0.018619
Epoch 011/20 | train RMSE 0.019140 | val RMSE 0.018626
Epoch 012/20 | train RMSE 0.019129 | val RMSE 0.018624
Epoch 013/20 | train RMSE 0.019150 | val RMSE 0.018609
Epoch 014/20 | train RMSE 0.019112 | val RMSE 0.018588
Epoch 015/20 | train RMSE 0.019057 | val RMSE 0.018643
Epoch 016/20 | train RMSE 0.019040 | val RMSE 0.018622
Epoch 017/20 | train RMSE 0.019040 | val RMSE 0.018620
Epoch 018/20 | train RMSE 0.019033 | val RMSE 0.018630
Epoch 019/

(FinancialLSTM(
   (lstm): LSTM(10, 20, num_layers=2, batch_first=True, dropout=0.1)
   (out_layer): Linear(in_features=20, out_features=10, bias=True)
 ),
 {'train_rmse': [0.0818770103041241,
   0.019730398273580953,
   0.019366795025816827,
   0.01933196497228108,
   0.019271441034023926,
   0.01927948380441017,
   0.019207251872835666,
   0.019181649363700773,
   0.019182975738887637,
   0.0191650368136689,
   0.019140203526004162,
   0.019128738348636348,
   0.019149582874220226,
   0.019111792200072468,
   0.019056566529375988,
   0.019040320150565407,
   0.019040417834380475,
   0.01903339028216176,
   0.019014494123211097,
   0.01900742658962917],
  'val_rmse': [0.01999052409716136,
   0.01860362123840988,
   0.018607941406164404,
   0.0185855364538043,
   0.0186405747913254,
   0.018589793451236902,
   0.01860572845295847,
   0.018603111710184937,
   0.018627379541245773,
   0.01861936357507977,
   0.018625677136308666,
   0.018623917907283728,
   0.018609221119010574,
   0.018

In [ ]:
class FinancialTransformerEncoder(nn.Module):
    def __init__(
        self,
        indim,
        out_dim,
        seq_len=20,
        d_model=64,
        nhead=4,
        num_layers=3,
        dim_feedforward=128,
        dropout=0.1,
    ):
        super().__init__()

        self.seq_len = seq_len
        self.indim = indim
        self.out_dim = out_dim
        self.d_model = d_model

        # project asset returns at each timestep -> model dimension
        self.input_proj = nn.Linear(indim, d_model)

        # learned positional embedding
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )

        # regression head: use last token representation
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, out_dim),
        )

        self._reset_parameters()

    def _reset_parameters(self):
        # PyTorch warns encoder layers are cloned with same initial params,
        # so we manually initialize everything.
        nn.init.normal_(self.pos_embedding, mean=0.0, std=0.02)

        for name, p in self.named_parameters():
            if p.dim() > 1 and "pos_embedding" not in name:
                nn.init.xavier_uniform_(p)
            elif p.dim() == 1 and "norm" not in name:
                nn.init.zeros_(p)

        # good defaults for layer norm
        for m in self.modules():
            if isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        """
        x: [batch, seq_len, indim]
        """
        bsz, T, _ = x.shape
        if T > self.seq_len:
            raise ValueError(f"Input seq_len={T} exceeds configured seq_len={self.seq_len}")

        x = self.input_proj(x)                     # [B, T, d_model]
        x = x + self.pos_embedding[:, :T, :]      # add positional information
        x = self.encoder(x)                       # [B, T, d_model]

        last_token = x[:, -1, :]                  # [B, d_model]
        out = self.head(last_token)               # [B, out_dim]
        return out

In [ ]:
window_size = 20
n_assets = 10
import copy

model = FinancialTransformerEncoder(
    indim=n_assets,
    out_dim=n_assets,
    seq_len=window_size,
    d_model=64,
    nhead=4,
    num_layers=3,
    dim_feedforward=128,
    dropout=0.1,
)

model, history = train(
    model,
    train_data_loader,
    val_data_loader=val_data_loader,
    epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
)

/tmp/ipykernel_19146/3250929715.py:36: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


Epoch 001/30 | train RMSE 0.179029 | val RMSE 0.022337
Epoch 002/30 | train RMSE 0.059300 | val RMSE 0.020309
Epoch 003/30 | train RMSE 0.050982 | val RMSE 0.020116
Epoch 004/30 | train RMSE 0.045415 | val RMSE 0.021795
Epoch 005/30 | train RMSE 0.041739 | val RMSE 0.020180
Epoch 006/30 | train RMSE 0.037398 | val RMSE 0.019836
Epoch 007/30 | train RMSE 0.034344 | val RMSE 0.019587
Epoch 008/30 | train RMSE 0.031623 | val RMSE 0.019855
Epoch 009/30 | train RMSE 0.029493 | val RMSE 0.019028
Epoch 010/30 | train RMSE 0.027830 | val RMSE 0.019215
Epoch 011/30 | train RMSE 0.026372 | val RMSE 0.019180
Epoch 012/30 | train RMSE 0.025160 | val RMSE 0.019437
Epoch 013/30 | train RMSE 0.024407 | val RMSE 0.018888
Epoch 014/30 | train RMSE 0.023524 | val RMSE 0.018839
Epoch 015/30 | train RMSE 0.022866 | val RMSE 0.019063
Epoch 016/30 | train RMSE 0.022447 | val RMSE 0.018929
Epoch 017/30 | train RMSE 0.022049 | val RMSE 0.018783
Epoch 018/30 | train RMSE 0.021662 | val RMSE 0.018905
Epoch 019/

In [6]:
# =========================================================
# 3) DECODER-ONLY TRANSFORMER BLOCK
# =========================================================

class DecoderBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=128, dropout=0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        # x: [B, T, d_model]

        # Pre-norm causal self-attention
        h = self.ln1(x)
        attn_out, _ = self.attn(
            h, h, h,
            attn_mask=attn_mask,     # [T, T], True means blocked
            need_weights=False
        )
        x = x + self.dropout1(attn_out)

        # Pre-norm feedforward
        h = self.ln2(x)
        x = x + self.dropout2(self.mlp(h))
        return x

# =========================================================
# 4) DECODER-ONLY TRANSFORMER MODEL
#    Outputs prediction for EVERY timestep in the input
#    shape in  : [B, T, n_assets]
#    shape out : [B, T, n_assets]
# =========================================================

class FinancialDecoderOnlyTransformer(nn.Module):
    def __init__(
        self,
        indim,
        out_dim,
        seq_len=20,
        d_model=64,
        nhead=4,
        num_layers=4,
        dim_feedforward=128,
        dropout=0.1,
    ):
        super().__init__()

        self.seq_len = seq_len
        self.indim = indim
        self.out_dim = out_dim
        self.d_model = d_model

        # project asset vector at each time step -> d_model
        self.input_proj = nn.Linear(indim, d_model)

        # learned positional embedding
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))

        # decoder-only stack
        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(d_model)

        # predict next-step returns for every position
        self.head = nn.Linear(d_model, out_dim)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.normal_(self.pos_embedding, mean=0.0, std=0.02)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def _causal_mask(self, T, device):
        # True means "not allowed to attend"
        return torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)

    def forward(self, x):
        # x: [B, T, indim]
        B, T, _ = x.shape
        if T > self.seq_len:
            raise ValueError(f"Input sequence length {T} exceeds configured seq_len={self.seq_len}")

        attn_mask = self._causal_mask(T, x.device)   # [T, T]

        x = self.input_proj(x)                       # [B, T, d_model]
        x = x + self.pos_embedding[:, :T, :]        # [B, T, d_model]

        for block in self.blocks:
            x = block(x, attn_mask=attn_mask)

        x = self.final_ln(x)
        out = self.head(x)                          # [B, T, out_dim]
        return out


In [7]:


class FinancialARSequenceDataset(Dataset):
    def __init__(self, numpy_dataset, window_size=20):
        self.data = numpy_dataset.astype(np.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        seq = self.data[idx : idx + self.window_size + 1]   # [window+1, n_assets]
        x = seq[:-1]                                        # [window, n_assets]
        y = seq[1:]                                         # [window, n_assets]
        return x, y

window_size = 20
batch_size = 64

train_dataset = FinancialARSequenceDataset(train_data, window_size=window_size)
val_dataset   = FinancialARSequenceDataset(val_data,   window_size=window_size)
test_dataset  = FinancialARSequenceDataset(test_data,  window_size=window_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)



In [11]:

import copy
# =========================================================
# 6) CREATE + TRAIN
# =========================================================

n_assets = len(tickers)

model = FinancialDecoderOnlyTransformer(
    indim=n_assets,
    out_dim=n_assets,
    seq_len=window_size,
    d_model=64,
    nhead=4,
    num_layers=4,
    dim_feedforward=128,
    dropout=0.1,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, history = train(
    model,
    train_loader,
    val_data_loader=val_loader,
    epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    grad_clip=1.0,
    device=device,
)

# =========================================================
# 7) EVALUATION EXAMPLE
# =========================================================

model.eval()
all_prediction = []
all_error = []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device).float()
        y = y.to(device).float()

        y_pred = model(x)                            # [B, T, D]
        all_prediction.append(y_pred.cpu().numpy())
        all_error.append((y_pred - y).cpu().numpy())

pred = np.concatenate(all_prediction, axis=0)       # [N, T, D]
err  = np.concatenate(all_error, axis=0)            # [N, T, D]
actual = pred - err                                 # [N, T, D]

print("pred shape  :", pred.shape)
print("actual shape:", actual.shape)
print("err shape   :", err.shape)

Epoch 001/30 | train RMSE 0.321705 | val RMSE 0.022996
Epoch 002/30 | train RMSE 0.134519 | val RMSE 0.023445
Epoch 003/30 | train RMSE 0.100666 | val RMSE 0.022833
Epoch 004/30 | train RMSE 0.075796 | val RMSE 0.020654
Epoch 005/30 | train RMSE 0.060397 | val RMSE 0.020830
Epoch 006/30 | train RMSE 0.051392 | val RMSE 0.021064
Epoch 007/30 | train RMSE 0.045532 | val RMSE 0.020598
Epoch 008/30 | train RMSE 0.041545 | val RMSE 0.020441
Epoch 009/30 | train RMSE 0.038661 | val RMSE 0.020450
Epoch 010/30 | train RMSE 0.036380 | val RMSE 0.019380
Epoch 011/30 | train RMSE 0.034459 | val RMSE 0.019754
Epoch 012/30 | train RMSE 0.033133 | val RMSE 0.019375
Epoch 013/30 | train RMSE 0.031666 | val RMSE 0.019231
Epoch 014/30 | train RMSE 0.030673 | val RMSE 0.019202
Epoch 015/30 | train RMSE 0.029629 | val RMSE 0.019871
Epoch 016/30 | train RMSE 0.028731 | val RMSE 0.019427
Epoch 017/30 | train RMSE 0.027970 | val RMSE 0.019274
Epoch 018/30 | train RMSE 0.027275 | val RMSE 0.019253
Epoch 019/